# A2: standalone hierarchical Bernoulli localisation

This is **Model A2 only**. It retains A1's binary top-3 frequency hit, SNR 1.25, 15 geometries,
two generators, paired phase/background design, and all arms. It changes the regression to allow
different lock effects by geometry and different probabilities for the lower and upper controls.
It also forecasts the **background alone through Chronos**, on exactly the backgrounds used for
the injected-tone arms. This calibration was absent from A1's background-truth check.

1. Upload this file to Colab and select a GPU runtime. Collection uses GPU when available;
   MCMC uses CPU/nutpie, with a PyMC CPU fallback. Running entirely on CPU is supported.
2. Keep `RUN_MODE = "PILOT"`, then Run all. If setup deliberately restarts the runtime,
   reconnect and Run all again. A2 uses its own Drive directory.
3. AUTO reuses `patchAliasing/pilots/model_A_localisation_bernoulli_v1` when compatible.
   With the completed A1 pilot this reuses **33,390 tone forecasts** and adds **90 background-only forecasts**.
   It copies the saved background arrays, not approximate regenerations. If no source exists, A2 collects its own data.
4. PILOT uses 3 backgrounds per generator, 4 chains, 3,000 retained draws and 2,000 tuning steps.
   Recovery uses 2,000 draws and 1,500 tuning steps. It cannot issue an empirical H1 verdict.
5. FULL requires a matching A2 PILOT PASS. Change only `RUN_MODE` to `"FULL"` in a clean runtime.
   FULL uses 100 backgrounds per generator and adds prior/link sensitivity.

The observed forecast horizon is 64 samples after a 480-sample context, at 512 Hz. Scoring uses
the raw median forecast, mean removal, a rectangular window, 8192 FFT points, and the top 3 local
maxima separated by at least 8 Hz. A target is hit within 1 Hz of any retained peak; flat forecasts
have no peak. No truth-hit filtering is applied.

## Declared model and interpretation

\[
h_i\sim\mathrm{Bernoulli}(p_i),\qquad
\mathrm{logit}(p_i)=\beta_{c[i]}+\gamma_{c[i]}L_i+\kappa_{c[i]}D_i+u_{k[i]}+u_{b[i]}.
\]

`L=1` for lock, otherwise 0. `D=-1,0,+1` for lower control, lock, upper control.
The configuration baseline has A1's centred overlap and log-patch-size covariates.
Both lock and side coefficients have Normal configuration hierarchies, with Student-t(4,0,0.5)
population means and HalfStudent-t(4,0.5) hierarchy scales. Baseline population scale is 1.5;
baseline covariate and group-scale priors retain scale 0.5. Harmonic and background offsets sum to zero.
The harmonic index remains A1's lock-site frequency rounded to 3 decimals.

`gamma_config[c]` compares lock log odds to the **midpoint of the two control log odds**.
Its odds ratio is relative to the geometric mean of control odds, not the odds of the pooled
control probability. `gamma + kappa` compares lock to the lower control, and `gamma - kappa`
compares lock to the upper control. All three contrasts are reported by geometry.

The primary finite-design estimand, `gamma_design`, gives each of the 15 registered geometries
equal weight. The hypermean `gamma_bar` is a different parameter. A design-average result does
not assert a common sign or magnitude for all geometries. Overlap/patch coefficients still concern
the baseline; no M1 mitigation verdict is issued.

Under valid FULL gates, support requires P(OR < 0.8) >= 0.95; practical equivalence requires
P(abs(log OR) < log(1.1)) >= 0.95. P(OR > 1.25) >= 0.95 is explicitly labelled **OPPOSITE DIRECTION**.
Otherwise the result is inconclusive. These thresholds concern the declared equal-geometry
log-odds estimand, not amplitude attenuation or a causal tone-response effect.

Counts pool only identical full predictor vectors, **keeping lo and hi separate**. This is an exact
Binomial evaluation of the Bernoulli likelihood, up to a data-only constant, and is checked numerically.
PILOT has 3,690 count groups; FULL has 123,000, without dropping trials. Conditional independence
of trials given the model effects is still an assumption to assess, not established by aggregation.


## 1. Setup
### 1.1 Locate the repository
Analysis code is embedded here; the frozen repository provides signal generators and the checkpoint loader.


In [2]:
import ipywidgets as widgets
from IPython.display import display

out = widgets.Output()
display(out)
with out:
    print("Widget Output funzionante")

Output()

In [3]:
import sys
import time
import ipywidgets
from rich.progress import track

print(sys.executable)
print("ipywidgets:", ipywidgets.__version__)

for _ in track(range(20), description="Test avanzamento"):
    time.sleep(0.1)

Output()

c:\Users\bonio\dev\patchAliasing\.venv\Scripts\python.exe
ipywidgets: 8.1.9


In [4]:
import importlib.util, os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
REPO_REVISION = "9d478e9a5aada82d138a47262ab5c2dc592911bc"
MARKER = Path("chronos/bayesian/probe_lib.py")
def on_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False
IS_COLAB = on_colab()
def find_repo():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(target)])
        subprocess.check_call(["git", "-C", str(target), "checkout", "--detach", REPO_REVISION])
    return target
REPO = find_repo()
BAYES_DIR = REPO / "chronos/bayesian"
sys.path.insert(0, str(BAYES_DIR)) if str(BAYES_DIR) not in sys.path else None
print("Repository:", REPO)


Repository: c:\Users\bonio\dev\patchAliasing


### 1.2 Environment
Colab installs the repository lock and verifies fresh imports after a deliberate restart. Local execution uses the active project environment. The posterior code supports both InferenceData and DataTree.


In [5]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A2_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf==1.8.1", "h5py==3.16.0"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this analysis.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            with tempfile.TemporaryDirectory() as temporary:
                requirements = Path(temporary) / "requirements.locked.txt"
                subprocess.check_call([UV, "export", "--frozen", "--no-dev", "--no-emit-project",
                                       "--no-hashes", "--output-file", str(requirements)], cwd=REPO)
                subprocess.check_call([UV, "pip", "install", "--python", sys.executable,
                                       "--reinstall-package", "numpy", "--reinstall-package", "scipy",
                                       "--requirement", str(requirements)])
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


Local runtime: dependency installation skipped; using the active environment.


### 1.3 Settings
AUTO looks for A1 arm tables and their manifests on Drive. `A2_SOURCE_RUN` may name another compatible run root. Existing A1 and A2 outputs use separate namespaces.


In [6]:
import gc, hashlib, inspect, json, platform, re, time
from pathlib import Path
from importlib import metadata as importlib_metadata
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import xarray as xr
import matplotlib.pyplot as plt
from scipy.special import expit, ndtr, logit, gammaln
from IPython.display import display
import checkpointing as cp
import probe_lib as pl
import model_loader as ml

RUN_MODE = "FULL"
RUN_ID = "model_A2_localisation_bernoulli_v2"
MODEL_VERSION = "A2-varying-lock-and-side-v1"
NOTEBOOK_VERSION = "A2-standalone-v1"
SEED = 42
TONE_SNR, TOP_K, TOL_HZ, NFFT, N_PHASE = 1.25, 3, 1.0, 8192, 10
MODELS, GENERATORS = list(pl.DELIVERABLE3_MODELS), tuple(pl.GENERATORS)
PRIOR_SCALE, BASELINE_SCALE, NU = .5, 1.5, 4
PRIOR_SCALES = (.25, .5, 1.)
SUPPORT_LOG_OR, ROPE_LOG_OR, PROB_CUTOFF = float(np.log(.8)), float(np.log(1.1)), .95
RHAT_MAX, PILOT_ESS_MIN, FULL_ESS_MIN = 1.01, 400, 1000
PPC_MIN_COVERAGE, SENSITIVITY_MAX_SPREAD, RECOVERY_MIN_COVERAGE = .90, .10, .80
if RUN_MODE not in {"PILOT", "FULL"}:
    raise ValueError("RUN_MODE must be PILOT or FULL")
IS_FULL = RUN_MODE == "FULL"
N_BG = 100 if IS_FULL else 3
DRAWS, TUNE, CHAINS = 10000, 2000, 4
CORES = max(1, min(CHAINS, os.cpu_count() or 1))
TARGET_ACCEPT = .95
RECOVERY_DRAWS, RECOVERY_TUNE = 2000, 1500
PPC_DRAWS, EFFECT_DRAWS = 800, 2000
BATCH_SIZE, BG_PER_SHARD, SPECTRAL_BATCH = 64, 10, 256
NUTS_BACKEND = "nutpie" if importlib.util.find_spec("nutpie") else "pymc"
COLLECTION_DEVICE = "auto"  # CUDA when available; Bayesian inference always uses CPU.
DATA_REUSE = "AUTO"  # OFF collects new arms and backgrounds in this run's namespace.
SOURCE_RUN_OVERRIDE = os.environ.get("A2_SOURCE_RUN", "")  # A1 run root, not its data subfolder.
if IS_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
default_root = Path("/content/drive/MyDrive/patchAliasing") if IS_COLAB else BAYES_DIR/"_run"
DRIVE_ROOT = Path(os.environ.get("A2_DRIVE_ROOT", str(default_root)))
OUTPUT_ROOT = DRIVE_ROOT/("full" if IS_FULL else "pilots")/RUN_ID
PILOT_RESULT_PATH = DRIVE_ROOT/"pilots"/RUN_ID/"final_verdict.json"
DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT = (OUTPUT_ROOT/name for name in ("data","checkpoints","figures"))
MANIFEST_PATH = OUTPUT_ROOT/"analysis_manifest.json"
for directory in (OUTPUT_ROOT, DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True,exist_ok=True)
pl.TONE_SNR = TONE_SNR
np.random.seed(SEED)
plt.style.use(next((s for s in ("arviz-whitegrid","seaborn-v0_8-whitegrid") if s in plt.style.available),"default"))
print("Mode:",RUN_MODE,"| MCMC:",NUTS_BACKEND,"CPU cores:",CORES)
print("PyMC / ArviZ / Python:",pm.__version__,az.__version__,platform.python_version())
print("Backgrounds per generator:",N_BG,"| draws/tune/chains:",DRAWS,TUNE,CHAINS)
print("Output:",OUTPUT_ROOT)


Mode: FULL | MCMC: pymc CPU cores: 4
PyMC / ArviZ / Python: 5.28.5 0.23.4 3.11.9
Backgrounds per generator: 100 | draws/tune/chains: 10000 2000 4
Output: c:\Users\bonio\dev\patchAliasing\chronos\bayesian\_run\full\model_A2_localisation_bernoulli_v2


## 2. Pipeline definitions
These definitions are fingerprinted before collecting or sampling. Source-data compatibility checks concern measurement and checkpoint provenance, not equality of the A1/A2 statistical models.


In [7]:
def frequency_design():
    rows = []
    for P, S in MODELS:
        for f_lock in pl.f_lock(P, S):
            delta = pl.control_offset(P, S, f_lock)
            if not np.isfinite(delta):
                continue
            phases = pl.phases_Sf(f_lock, N_PHASE)
            for phase_idx, phase in enumerate(phases):
                for role, f in (("lock", f_lock), ("lo", f_lock-delta), ("hi", f_lock+delta)):
                    rows.append(dict(model=pl.model_tag(P,S), P=P, S=S, overlap=(P-S)/P,
                                     f_lock=float(f_lock), delta=float(delta), phase_idx=phase_idx,
                                     phase=float(phase), role=role, f=float(f), is_lock=int(role=="lock")))
    return pd.DataFrame(rows)

def spectral_peaks(values):
    values = np.atleast_2d(np.asarray(values, dtype=float))
    if values.shape[1] != pl.PRED or not np.isfinite(values).all():
        raise ValueError("Expected finite forecast horizons only")
    pieces = []
    for start in range(0, len(values), SPECTRAL_BATCH):
        block = values[start:start+SPECTRAL_BATCH]
        peaks = pl.dominant_freqs(block, k=TOP_K, nfft=NFFT, band=pl.BAND)
        # A truly flat forecast has no spectral peak. Avoid argmax assigning the first band bin.
        peaks[np.ptp(block, axis=1) == 0] = np.nan
        pieces.append(peaks)
    return np.concatenate(pieces)

def hit_values(peaks, frequencies):
    return pl.localisation_hit(peaks, frequencies, tol=TOL_HZ)

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def logic_fingerprint(functions):
    # Code-object content fingerprints the executing functions without relying on notebook paths.
    # Exclude filenames/line numbers, which change between otherwise identical Colab executions.
    import types
    def normalise(value):
        if isinstance(value, types.CodeType):
            return dict(code=value.co_code.hex(), names=value.co_names, variables=value.co_varnames,
                        constants=[normalise(x) for x in value.co_consts],
                        argcount=value.co_argcount, kwonly=value.co_kwonlyargcount,
                        freevars=value.co_freevars, cellvars=value.co_cellvars)
        if isinstance(value, (tuple, list)):
            return [normalise(x) for x in value]
        if isinstance(value, (set, frozenset)):
            # Set display order depends on PYTHONHASHSEED; a clean kernel must keep the same hash.
            return {"set":sorted((normalise(x) for x in value), key=cp.canonical_json)}
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return repr(value)
    return cp.fingerprint({f.__name__:normalise(f.__code__) for f in functions})

def read_manifest():
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if current.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("Run manifest changed; do not combine different analyses")
    return current

def record_artifact(path):
    path = Path(path)
    manifest = read_manifest()
    manifest["artifacts"][path.relative_to(OUTPUT_ROOT).as_posix()] = {
        "sha256":cp.sha256_file(path), "bytes":path.stat().st_size}
    cp.atomic_json(MANIFEST_PATH, manifest)

def valid_artifact(path):
    path = Path(path)
    entry = read_manifest()["artifacts"].get(path.relative_to(OUTPUT_ROOT).as_posix())
    if path.is_file() != (entry is not None):
        raise ValueError(f"Untracked or missing artifact: {path}")
    if entry is None:
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"Artifact hash mismatch: {path}")
    return True

def save_table(path, frame):
    cp.atomic_parquet(path, frame)
    record_artifact(path)

def validate_arms(frame, design, n_bg=None):
    n_bg = N_BG if n_bg is None else n_bg
    keys = ["model", "generator", "bg_id", "f_lock", "phase_idx", "role"]
    if len(frame) != len(design)*len(GENERATORS)*n_bg or frame[keys].duplicated().any():
        raise ValueError("Incomplete/duplicate A-only arm design")
    if set(frame["generator"]) != set(GENERATORS) or set(frame["bg_id"]) != set(range(n_bg)):
        raise ValueError("Wrong background population")
    merged = frame.merge(design, on=["model", "f_lock", "phase_idx", "role"],
                         suffixes=("", "_expected"), how="left", validate="many_to_one")
    for name in ("P", "S", "overlap", "phase", "f", "delta", "is_lock"):
        if not np.allclose(merged[name], merged[name+"_expected"], rtol=0, atol=1e-9):
            raise ValueError(f"Arm metadata differs from the frozen design: {name}")
    sizes = frame.groupby(["generator", "bg_id"]).size()
    if len(sizes) != len(GENERATORS)*n_bg or not sizes.eq(len(design)).all():
        raise ValueError("Unequal or missing arm coverage per background")
    for name in ("h", "h_truth", "h_blind", "is_lock"):
        if not frame[name].isin([0,1]).all():
            raise ValueError(f"{name} must be binary")


### 2.1 Reuse, collection and matched background forecasts
Completed geometry/background shards are resumable. Reusing A1 copies both its validated arm rows and its exact stored backgrounds. The new baseline is a Chronos forecast of that same background context without an injected tone.


In [8]:
def source_manifest(source):
    path = Path(source["root"])/"analysis_manifest.json"
    if cp.sha256_file(path) != source["manifest_sha256"]:
        raise ValueError("Source manifest changed during this run")
    return json.loads(path.read_text(encoding="utf-8"))

def source_artifact(source, relative):
    path = Path(source["root"])/relative
    entry = source_manifest(source).get("artifacts",{}).get(relative)
    if entry is None or not path.is_file() or cp.sha256_file(path)!=entry["sha256"]:
        raise ValueError(f"Source artifact missing or changed: {path}")
    return path

def resolve_source():
    if DATA_REUSE not in {"AUTO","OFF"}:
        raise ValueError("DATA_REUSE must be AUTO or OFF")
    if DATA_REUSE=="OFF":
        return None
    candidates = [Path(SOURCE_RUN_OVERRIDE)] if SOURCE_RUN_OVERRIDE else [
        DRIVE_ROOT/"full/model_A_localisation_bernoulli_v1",
        *([DRIVE_ROOT/"pilots"/RUN_ID] if IS_FULL else []),
        DRIVE_ROOT/"pilots/model_A_localisation_bernoulli_v1"]
    for root in candidates:
        path = root/"data/A_arms.parquet"
        if not path.is_file():
            continue
        if root.resolve()==OUTPUT_ROOT.resolve():
            raise ValueError("The source and destination run must differ")
        manifest_path = root/"analysis_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        spec = manifest["analysis_spec"]
        if cp.fingerprint(spec)!=manifest.get("analysis_fingerprint"):
            raise ValueError("Source analysis fingerprint mismatch")
        science = spec["science"]
        if science.get("model") not in {"A-bernoulli-centred-zerosum-exact-counts-v1",MODEL_VERSION}:
            raise ValueError("Source is not a compatible A1/A2 localisation run")
        checks = dict(snr=TONE_SNR,top_k=TOP_K,tolerance_hz=TOL_HZ,nfft=NFFT,n_phase=N_PHASE,seed=SEED,scope="all_arms")
        for key,value in checks.items():
            if science.get(key)!=value:
                raise ValueError(f"Incompatible source measurement: {key}; select another source or DATA_REUSE='OFF'")
        if (cp.fingerprint(science.get("models"))!=cp.fingerprint(MODELS)
                or tuple(science.get("generators",[]))!=GENERATORS
                or science.get("checkpoints")!=CHECKPOINT_IDENTITIES):
            raise ValueError("Source geometry/generator/checkpoint population differs")
        for relative,digest in MEASUREMENT_HELPER_HASHES.items():
            if science.get("helper_hashes",{}).get(relative)!=digest:
                raise ValueError(f"Source measurement helper differs: {relative}")
        for package in ("numpy","torch","chronos-forecasting"):
            if spec.get("packages",{}).get(package)!=package_version(package):
                raise ValueError(f"Source environment differs for {package}; use its environment or DATA_REUSE='OFF'")
        source_n_bg = int(spec["n_bg"])
        if spec.get("run_mode") not in {"PILOT","FULL"} or source_n_bg<1:
            raise ValueError("Source is not a PILOT/FULL population")
        result = dict(root=str(root),manifest_sha256=cp.sha256_file(manifest_path),
                      analysis_fingerprint=manifest["analysis_fingerprint"],n_bg=min(N_BG,source_n_bg),
                      source_n_bg=source_n_bg,kind=science["model"])
        result["arms_sha256"] = cp.sha256_file(source_artifact(result,"data/A_arms.parquet"))
        # Stored backgrounds, not regenerated approximations, make the extra baseline paired.
        for generator in GENERATORS:
            for bg_id in range(result["n_bg"]):
                source_artifact(result,f"data/backgrounds/{generator}_{bg_id:03d}.npy")
        return result
    if SOURCE_RUN_OVERRIDE:
        raise FileNotFoundError(f"No data/A_arms.parquet under {SOURCE_RUN_OVERRIDE}")
    return None

def canonical_background(generator,bg_id):
    path = DATA_ROOT/"backgrounds"/f"{generator}_{bg_id:03d}.npy"
    if valid_artifact(path):
        values = np.load(path)
    else:
        if SOURCE is not None and bg_id<SOURCE["n_bg"]:
            values = np.load(source_artifact(SOURCE,f"data/backgrounds/{generator}_{bg_id:03d}.npy"))
        else:
            values = pl.background(generator,pl.CANON_LEN,10000+bg_id)
        cp.atomic_npy(path,values)
        record_artifact(path)
    if values.shape!=(pl.CANON_LEN,) or not np.isfinite(values).all():
        raise ValueError("Invalid background horizon")
    if abs(float(values.mean()))>1e-5 or abs(float(values.std())-1)>1e-5:
        raise ValueError("Expected a centred, unit-variance canonical background")
    return values

def collection_device():
    if COLLECTION_DEVICE in {"cpu","cuda"}:
        return COLLECTION_DEVICE
    if COLLECTION_DEVICE!="auto":
        raise ValueError("COLLECTION_DEVICE must be auto, cpu or cuda")
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"

def validate_calibration(frame):
    keys = ["model","generator","bg_id"]
    expected = {(pl.model_tag(P,S),g,b) for P,S in MODELS for g in GENERATORS for b in range(N_BG)}
    if frame[keys].duplicated().any() or set(map(tuple,frame[keys].to_numpy()))!=expected:
        raise ValueError("Incomplete/duplicate background forecasts")
    signals = frame[[f"forecast_{i}" for i in range(pl.PRED)]].to_numpy(float)
    peaks = spectral_peaks(signals)
    stored = frame[[f"background_f_hat_{i+1}" for i in range(TOP_K)]].to_numpy(float)
    np.testing.assert_allclose(stored,peaks,rtol=0,atol=1e-9,equal_nan=True)

def attach_calibration(arms,calibration):
    columns = [f"background_f_hat_{i+1}" for i in range(TOP_K)]
    clean = arms.drop(columns=[*columns,"h_background_forecast"],errors="ignore")
    frame = clean.merge(calibration[["model","generator","bg_id",*columns]],on=["model","generator","bg_id"],how="left",validate="many_to_one",indicator=True)
    if not frame._merge.eq("both").all():
        raise ValueError("Some arms lack a matched background-only forecast")
    frame = frame.drop(columns="_merge")
    frame["h_background_forecast"] = hit_values(frame[columns].to_numpy(float),frame.f.to_numpy(float))
    return frame

def collect_A2(design):
    merged_path,baseline_path = DATA_ROOT/"A_arms.parquet",DATA_ROOT/"background_forecasts.parquet"
    if valid_artifact(merged_path) and valid_artifact(baseline_path):
        arms,calibration = pd.read_parquet(merged_path),pd.read_parquet(baseline_path)
        validate_arms(arms,design)
        validate_calibration(calibration)
        expected = attach_calibration(arms,calibration)
        np.testing.assert_array_equal(expected.h_background_forecast,arms.h_background_forecast)
        return arms,calibration
    source_arms,source_baseline = None,None
    if SOURCE is not None:
        source_arms = pd.read_parquet(source_artifact(SOURCE,"data/A_arms.parquet"))
        source_arms = source_arms[source_arms.bg_id<SOURCE["n_bg"]].copy()
        validate_arms(source_arms,design,n_bg=SOURCE["n_bg"])
        if "data/background_forecasts.parquet" in source_manifest(SOURCE).get("artifacts",{}):
            source_baseline = pd.read_parquet(source_artifact(SOURCE,"data/background_forecasts.parquet"))
    device = collection_device()
    print("Chronos device:",device,"| reusable backgrounds per generator:",SOURCE["n_bg"] if SOURCE else 0)
    parts,baseline_parts = [],[]
    new_tone_forecasts,new_background_forecasts = 0,0
    start = time.time()
    for P,S in MODELS:
        tag = pl.model_tag(P,S)
        base = design[design.model.eq(tag)].reset_index(drop=True)
        probe = None
        try:
            for generator in GENERATORS:
                for first in range(0,N_BG,BG_PER_SHARD):
                    arm_path = DATA_ROOT/"raw"/f"arms_{tag}_{generator}_{first:03d}.parquet"
                    cal_path = DATA_ROOT/"raw"/f"baseline_{tag}_{generator}_{first:03d}.parquet"
                    arms_done,cal_done = valid_artifact(arm_path),valid_artifact(cal_path)
                    if arms_done and cal_done:
                        parts.append(pd.read_parquet(arm_path)); baseline_parts.append(pd.read_parquet(cal_path))
                        continue
                    block_arms,block_cal = [],[]
                    for bg_id in range(first,min(first+BG_PER_SHARD,N_BG)):
                        bg = canonical_background(generator,bg_id)
                        reusable = SOURCE is not None and bg_id<SOURCE["n_bg"]
                        if not arms_done:
                            if reusable:
                                meta = source_arms[source_arms.model.eq(tag)&source_arms.generator.eq(generator)&source_arms.bg_id.eq(bg_id)].copy()
                            else:
                                if probe is None:
                                    probe = pl.Probe(P,S,device=device,batch_size=BATCH_SIZE)
                                    if probe.checkpoint_identity!=CHECKPOINT_IDENTITIES[tag]: raise ValueError("Checkpoint changed")
                                meta = base.copy()
                                meta["generator"],meta["bg_id"] = generator,bg_id
                                signals = np.stack([pl.build_context(bg,row.f,row.phase,pl.CANON_LEN) for row in base.itertuples(index=False)])
                                predicted = probe.forecast(signals[:,:pl.CTX])
                                peaks,truth_peaks = spectral_peaks(predicted),spectral_peaks(signals[:,pl.CTX:])
                                meta["h"],meta["h_truth"] = hit_values(peaks,meta.f),hit_values(truth_peaks,meta.f)
                                meta["h_blind"] = hit_values(np.repeat(spectral_peaks(bg[pl.CTX:]),len(meta),axis=0),meta.f)
                                for j in range(TOP_K):
                                    meta[f"f_hat_{j+1}"],meta[f"truth_f_hat_{j+1}"] = peaks[:,j],truth_peaks[:,j]
                                new_tone_forecasts += len(meta)
                            block_arms.append(meta)
                        if not cal_done:
                            reused_cal = None
                            if reusable and source_baseline is not None:
                                selected = source_baseline[source_baseline.model.eq(tag)&source_baseline.generator.eq(generator)&source_baseline.bg_id.eq(bg_id)]
                                if len(selected)!=1: raise ValueError("Source baseline row missing or duplicate")
                                reused_cal = selected.iloc[0].to_dict()
                            if reused_cal is None:
                                if probe is None:
                                    probe = pl.Probe(P,S,device=device,batch_size=BATCH_SIZE)
                                    if probe.checkpoint_identity!=CHECKPOINT_IDENTITIES[tag]: raise ValueError("Checkpoint changed")
                                # Critical: pass the same background context through Chronos, with no tone.
                                predicted_bg = np.asarray(probe.forecast(bg[None,:pl.CTX]),float)
                                peaks_bg = spectral_peaks(predicted_bg)[0]
                                reused_cal = dict(model=tag,generator=generator,bg_id=bg_id)
                                reused_cal.update({f"forecast_{i}":float(value) for i,value in enumerate(predicted_bg[0])})
                                reused_cal.update({f"background_f_hat_{j+1}":float(value) for j,value in enumerate(peaks_bg)})
                                new_background_forecasts += 1
                            block_cal.append(reused_cal)
                    if not arms_done: save_table(arm_path,pd.concat(block_arms,ignore_index=True))
                    if not cal_done: save_table(cal_path,pd.DataFrame(block_cal))
                    parts.append(pd.read_parquet(arm_path)); baseline_parts.append(pd.read_parquet(cal_path))
                    print(f"{tag} {generator} bg {first}:{min(first+BG_PER_SHARD,N_BG)} saved; {(time.time()-start)/60:.1f} min")
        finally:
            if probe is not None: probe.close()
            gc.collect()
    arms,calibration = pd.concat(parts,ignore_index=True),pd.concat(baseline_parts,ignore_index=True)
    validate_arms(arms,design)
    validate_calibration(calibration)
    arms = attach_calibration(arms,calibration)
    save_table(baseline_path,calibration)
    save_table(merged_path,arms)
    print("New tone/background-only forecasts:",new_tone_forecasts,new_background_forecasts)
    return arms,calibration

def calibration_summary(arms,columns):
    result = arms.groupby(columns,observed=True).agg(
        n=("h","size"),forecast_hit=("h","mean"),true_future_hit=("h_truth","mean"),
        background_true_future_hit=("h_blind","mean"),background_forecast_hit=("h_background_forecast","mean")).reset_index()
    result["tone_minus_background_forecast"] = result.forecast_hit-result.background_forecast_hit
    result["truth_minus_background_truth"] = result.true_future_hit-result.background_true_future_hit
    return result


### 2.2 A2 model and exact role-separated counts
The model remains Bernoulli. Hierarchical lock and side effects allow geometry-dependent behavior while sharing information. No per-observation probability array is retained in the posterior.


In [9]:
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))

def _overlap_scaled(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred, scaled patch overlap O = (P-S)/P; one unit = 0.5 of overlap."""
    O = df.groupby("model")["overlap"].first().reindex(cfg_levels).to_numpy(float)
    return (O - O.mean()) / 0.5

def _logP_centred(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred log patch size.

    Deliverable 3, H1: "The overlap enters as a ratio and the patch size as log P, because the
    patch grid has spacing fs/P: equal steps in log P are then equal ratios of spacing." On a raw-P
    scale one coefficient would make 8->16 and 16->24 the same change, which the geometry does not.
    """
    Pv = df.groupby("model")["P"].first().reindex(cfg_levels).to_numpy(float)
    lp = np.log(Pv)
    return lp - lp.mean()

def model_A2(frame, scale=PRIOR_SCALE, baseline_scale=BASELINE_SCALE, nu=NU,
             link="logit", encoding="counts"):
    """Bernoulli A2; counts pool only rows with identical full predictor vectors.

    eta = beta_config + gamma_config*is_lock + kappa_config*side + u_harm + u_bg
    side = -1 (lo), 0 (lock), +1 (hi).
    gamma_config is the lock log odds minus the midpoint of the two control log odds.
    gamma_design is the equally weighted mean over the 15 observed configurations.
    """
    if link not in {"logit","probit"} or encoding not in {"counts","bernoulli"}:
        raise ValueError("Unknown link or likelihood encoding")
    ci,cl = _codes(frame.model)
    hi,hl = _codes(frame.f_lock.round(3).astype(str))
    bi,bl = _codes(frame.generator+"#"+frame.bg_id.astype(str))
    expected_side = frame.role.map({"lo":-1,"lock":0,"hi":1})
    if expected_side.isna().any() or not frame.is_lock.eq(frame.role.eq("lock").astype(int)).all():
        raise ValueError("Invalid role/lock encoding")
    if "side" in frame and not frame.side.eq(expected_side).all():
        raise ValueError("Side differs from the registered role encoding")
    y = frame["hits" if encoding=="counts" else "h"].to_numpy(float)
    n = frame.n_trials.to_numpy(float) if encoding=="counts" else np.ones(len(frame))
    if not (len(frame) and np.isfinite(y).all() and np.isfinite(n).all()
            and np.equal(y,np.floor(y)).all() and np.equal(n,np.floor(n)).all()
            and (n>=1).all() and (y>=0).all() and (y<=n).all()):
        raise ValueError("Invalid finite integer counts")
    coords = dict(config=cl,harmonic=hl,background=bl,obs=np.arange(len(frame)))
    with pm.Model(coords=coords) as model:
        beta_bar = pm.StudentT("beta_bar",nu=nu,mu=0,sigma=baseline_scale)
        delta_O = pm.StudentT("delta_O",nu=nu,mu=0,sigma=scale)
        delta_P = pm.StudentT("delta_P",nu=nu,mu=0,sigma=scale)
        tau = pm.HalfStudentT("tau",nu=nu,sigma=scale)
        beta = pm.Normal("beta",mu=beta_bar+delta_O*_overlap_scaled(frame,cl)+delta_P*_logP_centred(frame,cl),sigma=tau,dims="config")
        gamma_bar = pm.StudentT("gamma_bar",nu=nu,mu=0,sigma=scale)
        sigma_gamma = pm.HalfStudentT("sigma_gamma",nu=nu,sigma=scale)
        gamma = pm.Normal("gamma_config",mu=gamma_bar,sigma=sigma_gamma,dims="config")
        kappa_bar = pm.StudentT("kappa_bar",nu=nu,mu=0,sigma=scale)
        sigma_kappa = pm.HalfStudentT("sigma_kappa",nu=nu,sigma=scale)
        kappa = pm.Normal("kappa_config",mu=kappa_bar,sigma=sigma_kappa,dims="config")
        sigma_harm = pm.HalfStudentT("sigma_harm",nu=nu,sigma=scale)
        harm = pm.ZeroSumNormal("u_harm",sigma=sigma_harm,dims="harmonic")
        sigma_bg = pm.HalfStudentT("sigma_bg",nu=nu,sigma=scale)
        bg = pm.ZeroSumNormal("u_bg",sigma=sigma_bg,dims="background")
        eta = beta[ci]+gamma[ci]*frame.is_lock.to_numpy(float)+kappa[ci]*expected_side.to_numpy(float)+harm[hi]+bg[bi]
        parameter = {"logit_p":eta} if link=="logit" else {"p":pm.math.clip(.5*(1+pm.math.erf(eta/np.sqrt(2.))),1e-9,1-1e-9)}
        if encoding=="counts":
            pm.Binomial("hits",n=n.astype(int),observed=y.astype(int),dims="obs",**parameter)
        else:
            pm.Bernoulli("h",observed=y.astype(int),dims="obs",**parameter)
        if link=="logit":
            pm.Deterministic("gamma_design",pm.math.mean(gamma))
            pm.Deterministic("odds_ratio_design",pm.math.exp(pm.math.mean(gamma)))
    return model

def aggregate_trials(frame):
    keys = ["model","P","S","overlap","generator","bg_id","f_lock","role","is_lock"]
    result = frame.groupby(keys,observed=True,sort=True).agg(
        hits=("h","sum"),n_trials=("h","size"),truth_hits=("h_truth","sum"),
        background_truth_hits=("h_blind","sum"),background_forecast_hits=("h_background_forecast","sum")).reset_index()
    result["side"] = result.role.map({"lo":-1,"lock":0,"hi":1}).astype(int)
    for name in ("hits","n_trials","truth_hits","background_truth_hits","background_forecast_hits"):
        result[name] = result[name].astype(np.int64)
    assert result.n_trials.sum()==len(frame) and result.hits.sum()==frame.h.sum()
    return result


### 2.3 Inference, predictive checks and reporting
CPU MCMC, immutable fit/data fingerprints, atomic checkpoints, and generic posterior-group membership support the Colab DataTree format. Probit sensitivity is transformed onto the same declared log-odds scale.


In [10]:
def fit_or_load(label,frame,scale=PRIOR_SCALE,link="logit",draws=None,tune=None):
    path = CHECKPOINT_ROOT/f"{label}.nc"
    fields = ["model","generator","bg_id","f_lock","role","hits","n_trials"]
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(frame[fields],index=False).to_numpy().tobytes()).hexdigest()
    fit_spec = dict(model=MODEL_VERSION,data_sha256=data_hash,scale=scale,link=link,
                    draws=draws or DRAWS,tune=tune or TUNE,chains=CHAINS,target_accept=TARGET_ACCEPT)
    fit_hash = cp.fingerprint(fit_spec)
    if valid_artifact(path):
        idata = az.from_netcdf(path)
        idata.load()
        idata.close()
        if idata.attrs.get("fit_fingerprint")!=fit_hash:
            raise ValueError(f"Cached fit data/settings differ: {label}")
        return idata
    unit = 1.6 if link=="probit" else 1.
    model = model_A2(frame,scale=scale/unit,baseline_scale=BASELINE_SCALE/unit,link=link)
    start = time.time()
    print(f"{label}: {len(frame):,} groups / {frame.n_trials.sum():,} trials; "
          f"draws={draws or DRAWS}, tune={tune or TUNE}, chains={CHAINS}, cores={CORES}")
    with model:
        if not np.isfinite(model.compile_logp()(model.initial_point())):
            raise ValueError("Non-finite initial log probability")
        idata = pm.sample(draws=draws or DRAWS,tune=tune or TUNE,chains=CHAINS,cores=CORES,
                          target_accept=TARGET_ACCEPT,random_seed=SEED,progressbar=True,
                          nuts_sampler=NUTS_BACKEND,idata_kwargs={"log_likelihood":False})
    # Membership works with both legacy ArviZ InferenceData and xarray.DataTree.
    if "log_likelihood" in idata or "p" in idata.posterior:
        raise AssertionError("Observation-sized posterior output was unexpectedly retained")
    idata.attrs["fit_fingerprint"],idata.attrs["model_version"] = fit_hash,MODEL_VERSION
    cp.atomic_netcdf(path,idata)
    record_artifact(path)
    print(f"Saved {path.name} in {(time.time()-start)/60:.1f} minutes")
    return idata

def diagnostics_for(idata,ess_min):
    table = az.summary(idata,kind="diagnostics",round_to="none")
    table = table[["r_hat","ess_bulk","ess_tail"]].apply(pd.to_numeric,errors="coerce")
    result = dict(max_rhat=float(table.r_hat.max()),min_ess_bulk=float(table.ess_bulk.min()),
                  min_ess_tail=float(table.ess_tail.min()),divergences=int(np.asarray(idata.sample_stats["diverging"]).sum()))
    result["passed"] = bool(np.isfinite(table.to_numpy(float)).all() and (table.r_hat<RHAT_MAX).all()
                            and (table.ess_bulk>ess_min).all() and (table.ess_tail>ess_min).all() and result["divergences"]==0)
    return result,table

def posterior_array(idata,name,dimension=None):
    values = np.asarray(idata.posterior[name].transpose("chain","draw",*([dimension] if dimension else [])),float)
    return values.reshape(-1,values.shape[-1]) if dimension else values.ravel()

def posterior_components(idata,frame):
    def codes(values,dimension):
        lookup = {str(value):i for i,value in enumerate(idata.posterior.coords[dimension].values)}
        result = np.array([lookup.get(str(value),-1) for value in values],int)
        if (result<0).any(): raise ValueError(f"Unknown posterior {dimension} level")
        return result
    return dict(beta=posterior_array(idata,"beta","config"),gamma=posterior_array(idata,"gamma_config","config"),
                kappa=posterior_array(idata,"kappa_config","config"),harm=posterior_array(idata,"u_harm","harmonic"),
                bg=posterior_array(idata,"u_bg","background"),ci=codes(frame.model,"config"),
                hi=codes(frame.f_lock.round(3).astype(str),"harmonic"),bi=codes(frame.generator+"#"+frame.bg_id.astype(str),"background"))

def probability(eta,link):
    return expit(eta) if link=="logit" else np.clip(ndtr(eta),1e-9,1-1e-9)

def effect_summary(values):
    low,median,high = np.quantile(values,[.025,.5,.975])
    return dict(median_log_or=float(median),eti_low_log_or=float(low),eti_high_log_or=float(high),
                median_odds_ratio=float(np.exp(median)),p_support=float(np.mean(values<SUPPORT_LOG_OR)),
                p_equivalence=float(np.mean(np.abs(values)<ROPE_LOG_OR)),
                p_opposite=float(np.mean(values>-SUPPORT_LOG_OR)),p_positive=float(np.mean(values>0)))

def comparable_effects(idata,frame,link="logit"):
    """Per-geometry log odds against the midpoint of the two control log odds."""
    if link=="logit":
        return posterior_array(idata,"gamma_config","config")
    lock_rows = frame[frame.role.eq("lock")].reset_index(drop=True)
    arrays = posterior_components(idata,lock_rows)
    n = lock_rows.n_trials.to_numpy(float)
    weights = n/np.bincount(arrays["ci"],weights=n)[arrays["ci"]]
    selected = np.linspace(0,len(arrays["beta"])-1,min(EFFECT_DRAWS,len(arrays["beta"]))).astype(int)
    results = []
    for i in selected:
        base = arrays["beta"][i,arrays["ci"]]+arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]]
        gamma,kappa = arrays["gamma"][i,arrays["ci"]],arrays["kappa"][i,arrays["ci"]]
        difference = logit(probability(base+gamma,link))-.5*(logit(probability(base-kappa,link))+logit(probability(base+kappa,link)))
        results.append(np.bincount(arrays["ci"],weights=weights*difference,minlength=arrays["beta"].shape[1]))
    return np.asarray(results)

def geometry_effect_table(idata,frame):
    gamma,kappa = posterior_array(idata,"gamma_config","config"),posterior_array(idata,"kappa_config","config")
    rows = []
    for i,name in enumerate(idata.posterior.coords["config"].values):
        row = dict(model=str(name),**effect_summary(gamma[:,i]))
        for label,values in (("lock_vs_lo",gamma[:,i]+kappa[:,i]),("lock_vs_hi",gamma[:,i]-kappa[:,i]),("side",kappa[:,i])):
            lo,med,hi = np.quantile(values,[.025,.5,.975])
            row.update({label+"_median_log_or":float(med),label+"_low_log_or":float(lo),label+"_high_log_or":float(hi)})
        rows.append(row)
    return pd.DataFrame(rows)

def predictive_check(idata,frame,seed=SEED+600):
    arrays,rng = posterior_components(idata,frame),np.random.default_rng(seed)
    selected = np.linspace(0,len(arrays["beta"])-1,min(PPC_DRAWS,len(arrays["beta"]))).astype(int)
    specs = []
    for check_set,arm_column in (("A1_comparable","is_lock"),("separate_roles","role")):
        for dimension in ("model","generator"):
            for (level,arm),positions in frame.groupby([dimension,arm_column],observed=True).indices.items():
                specs.append(dict(check_set=check_set,dimension=dimension,level=str(level),arm=str(arm),
                                  key=f"{dimension}={level}|{arm_column}={arm}",positions=np.asarray(positions,int)))
    simulated_rates = [[] for _ in specs]
    n = frame.n_trials.to_numpy(int)
    for i in selected:
        eta = (arrays["beta"][i,arrays["ci"]]+arrays["gamma"][i,arrays["ci"]]*frame.is_lock.to_numpy(float)
               +arrays["kappa"][i,arrays["ci"]]*frame.side.to_numpy(float)
               +arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]])
        replicate = rng.binomial(n,expit(eta))
        for samples,spec in zip(simulated_rates,specs):
            pos = spec["positions"]
            samples.append(replicate[pos].sum()/n[pos].sum())
    rows = []
    for samples,spec in zip(simulated_rates,specs):
        pos = spec["positions"]
        observed = float(frame.hits.to_numpy()[pos].sum()/n[pos].sum())
        low,high = np.quantile(samples,[.025,.975])
        rows.append({**{k:v for k,v in spec.items() if k!="positions"},"n_trials":int(n[pos].sum()),
                     "observed_rate":observed,"rep_low":float(low),"rep_high":float(high),"passed":bool(low<=observed<=high)})
    return pd.DataFrame(rows)

def comparison_to_A1(ppc):
    if SOURCE is None or SOURCE["kind"]!="A-bernoulli-centred-zerosum-exact-counts-v1" or SOURCE["source_n_bg"]!=N_BG:
        return pd.DataFrame()
    manifest = source_manifest(SOURCE)
    if "primary_ppc.parquet" not in manifest.get("artifacts",{}):
        return pd.DataFrame()
    old = pd.read_parquet(source_artifact(SOURCE,"primary_ppc.parquet"))
    keys = []
    for value in old.stratum:
        match = re.fullmatch(r"\(\('(model|generator)', '([^']+)'\), \('is_lock', (?:np\.\w+\()?([01])\)?\)\)",str(value))
        if match is None: raise ValueError(f"Unrecognised A1 PPC key: {value}")
        dimension,level,arm = match.groups()
        keys.append(f"{dimension}={level}|is_lock={arm}")
    old["key"] = keys
    new = ppc[ppc.check_set.eq("A1_comparable")]
    if set(old.key)!=set(new.key) or old.key.duplicated().any(): raise ValueError("A1/A2 PPC strata differ")
    result = new.merge(old[["key","n_trials","observed_rate","rep_low","rep_high","passed"]],on="key",suffixes=("_A2","_A1"),validate="one_to_one")
    if not result.n_trials_A2.eq(result.n_trials_A1).all(): raise ValueError("A1/A2 PPC trial counts differ")
    np.testing.assert_allclose(result.observed_rate_A2,result.observed_rate_A1,atol=1e-12,rtol=0)
    return result

def recovery_data(groups):
    out = groups[groups.bg_id<3].copy().reset_index(drop=True)
    rng = np.random.default_rng(SEED+300)
    ci,cl = _codes(out.model)
    hi,hl = _codes(out.f_lock.round(3).astype(str))
    bi,bl = _codes(out.generator+"#"+out.bg_id.astype(str))
    truth = dict(beta_bar=-.4,delta_O=.2,delta_P=-.15,gamma_bar=-.25,kappa_bar=-.2)
    beta = rng.normal(truth["beta_bar"]+truth["delta_O"]*_overlap_scaled(out,cl)+truth["delta_P"]*_logP_centred(out,cl),.3)
    gamma,kappa = rng.normal(truth["gamma_bar"],.65,len(cl)),rng.normal(truth["kappa_bar"],.3,len(cl))
    harm = rng.normal(0,.4,len(hl)); harm-=harm.mean()
    bg = rng.normal(0,.2,len(bl)); bg-=bg.mean()
    eta = beta[ci]+gamma[ci]*out.is_lock.to_numpy()+kappa[ci]*out.side.to_numpy()+harm[hi]+bg[bi]
    out["hits"] = rng.binomial(out.n_trials.to_numpy(int),expit(eta))
    truth["gamma_design"] = float(gamma.mean())
    rows = [dict(parameter=name,variable=name,coordinate=None,truth=value) for name,value in truth.items()]
    rows += [dict(parameter=f"{name}[{level}]",variable=name,coordinate=level,truth=float(value))
             for name,values in (("gamma_config",gamma),("kappa_config",kappa)) for level,value in zip(cl,values)]
    return out,pd.DataFrame(rows)

def verdict_from(gate_ok,effect):
    if not gate_ok: return "NOT REPORTABLE"
    if effect["p_support"]>=PROB_CUTOFF: return "SUPPORTED"
    if effect["p_equivalence"]>=PROB_CUTOFF: return "PRACTICALLY EQUIVALENT"
    if effect["p_opposite"]>=PROB_CUTOFF: return "OPPOSITE DIRECTION"
    return "INCONCLUSIVE"


## 3. Preflight
Inspect reusable data and the number of additional forecasts before continuing. A matching A2 pilot is required for FULL. Synthetic tones here check the instrument only.


In [11]:
design = frequency_design()
CHECKPOINT_IDENTITIES = {pl.model_tag(P,S):ml.checkpoint_identity(P,S) for P,S in MODELS}
paths = [BAYES_DIR/name for name in ("probe_lib.py","model_loader.py","checkpointing.py")]
paths += sorted((REPO/"chronos/data/synthetic").rglob("*.py"))
MEASUREMENT_HELPER_HASHES = {path.relative_to(REPO).as_posix():cp.sha256_file(path) for path in paths}
SOURCE = resolve_source()
test_f = np.array([32.,64.,96.])
assert hit_values(spectral_peaks(np.stack([pl.make_tone(f,.37,pl.PRED) for f in test_f])),test_f).all()
assert np.isnan(spectral_peaks(np.zeros((1,pl.PRED)))).all()
assert not hit_values(np.full((1,TOP_K),np.nan),[32.]).any()
functions = [value for name,value in list(globals().items()) if inspect.isfunction(value) and value.__module__=="__main__" and name!="display"]
SCIENCE_SPEC = dict(notebook=NOTEBOOK_VERSION,model=MODEL_VERSION,models=MODELS,generators=GENERATORS,
                    snr=TONE_SNR,top_k=TOP_K,tolerance_hz=TOL_HZ,nfft=NFFT,n_phase=N_PHASE,seed=SEED,scope="all_arms",
                    prior_scale=PRIOR_SCALE,baseline_scale=BASELINE_SCALE,nu=NU,prior_scales=PRIOR_SCALES,
                    hierarchy="centred beta, gamma and kappa; zero-sum harmonic and background",
                    estimand="equal-geometry mean log odds vs midpoint of lo/hi log odds",side_coding={"lo":-1,"lock":0,"hi":1},
                    likelihood_encoding="exact_role_separated_binomial_counts",support_log_or=SUPPORT_LOG_OR,
                    rope_log_or=ROPE_LOG_OR,opposite_log_or=-SUPPORT_LOG_OR,cutoff=PROB_CUTOFF,
                    rhat_max=RHAT_MAX,ess_pilot=PILOT_ESS_MIN,ess_full=FULL_ESS_MIN,
                    ppc_min_coverage=PPC_MIN_COVERAGE,sensitivity_max_spread=SENSITIVITY_MAX_SPREAD,
                    recovery_min_coverage=RECOVERY_MIN_COVERAGE,
                    design_hash=cp.fingerprint(design.to_dict("records")),helper_hashes=MEASUREMENT_HELPER_HASHES,
                    checkpoints=CHECKPOINT_IDENTITIES,executing_logic_sha256=logic_fingerprint(functions))
CORE_FINGERPRINT = cp.fingerprint(SCIENCE_SPEC)
if IS_FULL:
    if not PILOT_RESULT_PATH.is_file(): raise FileNotFoundError(f"Run A2 PILOT first: {PILOT_RESULT_PATH}")
    pilot = json.loads(PILOT_RESULT_PATH.read_text(encoding="utf-8"))
    if pilot.get("core_fingerprint")!=CORE_FINGERPRINT or pilot.get("pilot_route_ok") is not True:
        raise ValueError("FULL requires a matching A2 PILOT PASS")
ANALYSIS_SPEC = dict(science=SCIENCE_SPEC,run_id=RUN_ID,run_mode=RUN_MODE,n_bg=N_BG,source=SOURCE,
                     draws=DRAWS,tune=TUNE,chains=CHAINS,target_accept=TARGET_ACCEPT,
                     recovery_draws=RECOVERY_DRAWS,recovery_tune=RECOVERY_TUNE,ppc_draws=PPC_DRAWS,effect_draws=EFFECT_DRAWS,
                     backend=NUTS_BACKEND,python=platform.python_version(),
                     packages={name:package_version(name) for name in ("numpy","scipy","pandas","pymc","arviz","nutpie","pytensor","torch","chronos-forecasting")})
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
if MANIFEST_PATH.is_file():
    previous = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if previous.get("analysis_fingerprint")!=ANALYSIS_FINGERPRINT:
        raise ValueError("Code/settings/source changed. Choose a new RUN_ID; existing runs are never auto-healed.")
else:
    cp.atomic_json(MANIFEST_PATH,dict(analysis_fingerprint=ANALYSIS_FINGERPRINT,analysis_spec=ANALYSIS_SPEC,artifacts={}))
total = len(design)*len(GENERATORS)*N_BG
reusable = len(design)*len(GENERATORS)*(SOURCE["n_bg"] if SOURCE else 0)
baseline_total = len(MODELS)*len(GENERATORS)*N_BG
source_has_calibration = SOURCE is not None and "data/background_forecasts.parquet" in source_manifest(SOURCE).get("artifacts",{})
baseline_reusable = len(MODELS)*len(GENERATORS)*SOURCE["n_bg"] if source_has_calibration else 0
print("Source:",SOURCE["root"] if SOURCE else "fresh collection")
print(f"A2 trials: {total:,}; reusable tone forecasts: {reusable:,}; additional tone forecasts: {total-reusable:,}")
print(f"Background-only Chronos forecasts: {baseline_total:,}; reusable: {baseline_reusable:,}; additional: {baseline_total-baseline_reusable:,}")
print("Observed probability/count arrays are not stored once per posterior draw.")
print("Output:",OUTPUT_ROOT)


Source: c:\Users\bonio\dev\patchAliasing\chronos\bayesian\_run\pilots\model_A2_localisation_bernoulli_v2
A2 trials: 1,113,000; reusable tone forecasts: 33,390; additional tone forecasts: 1,079,610
Background-only Chronos forecasts: 3,000; reusable: 90; additional: 2,910
Observed probability/count arrays are not stored once per posterior draw.
Output: c:\Users\bonio\dev\patchAliasing\chronos\bayesian\_run\full\model_A2_localisation_bernoulli_v2


## 4. Collect A2 arms and calibration
Calibration rates are descriptive on the registered trial weights. A frequent background-only hit can indicate spontaneous forecast peaks at the target. It is not evidence that an injected tone was preserved. Background-truth and background-forecast rates are kept distinct.


In [12]:
arms,background_forecasts = collect_A2(design)
validate_arms(arms,design)
groups = aggregate_trials(arms)
save_table(DATA_ROOT/"A_counts.parquet",groups)
if arms.h.nunique()<2: raise ValueError("All tone forecasts have the same hit outcome; inspect the instrument before fitting")
rates = calibration_summary(arms,["generator","role"])
geometry_rates = calibration_summary(arms,["model","role"])
save_table(DATA_ROOT/"calibration_by_generator.parquet",rates)
save_table(DATA_ROOT/"calibration_by_geometry.parquet",geometry_rates)
MEASUREMENT_OK = bool((rates.truth_minus_background_truth>0).all())
CALIBRATION_COMPLETE = len(background_forecasts)==len(MODELS)*len(GENERATORS)*N_BG
print(f"{len(arms):,} trials -> {len(groups):,} role-separated count groups")
display(rates)
display(geometry_rates)
print("Instrument separation:",MEASUREMENT_OK,"| paired background forecasts complete:",CALIBRATION_COMPLETE)


Chronos device: cpu | reusable backgrounds per generator: 3
p16-s16 kernelsynth bg 60:70 saved; 0.1 min
p16-s16 kernelsynth bg 70:80 saved; 0.2 min
p16-s16 kernelsynth bg 80:90 saved; 0.2 min
p16-s16 kernelsynth bg 90:100 saved; 0.3 min
p24-s8 tsmixup bg 0:10 saved; 0.4 min
p24-s8 tsmixup bg 10:20 saved; 0.5 min
p24-s8 tsmixup bg 20:30 saved; 0.6 min
p24-s8 tsmixup bg 30:40 saved; 0.8 min
p24-s8 tsmixup bg 40:50 saved; 0.9 min
p24-s8 tsmixup bg 50:60 saved; 1.0 min
p24-s8 tsmixup bg 60:70 saved; 1.1 min
p24-s8 tsmixup bg 70:80 saved; 1.3 min
p24-s8 tsmixup bg 80:90 saved; 1.4 min
p24-s8 tsmixup bg 90:100 saved; 1.5 min
p24-s8 kernelsynth bg 0:10 saved; 1.6 min
p24-s8 kernelsynth bg 10:20 saved; 1.7 min
p24-s8 kernelsynth bg 20:30 saved; 1.9 min
p24-s8 kernelsynth bg 30:40 saved; 2.0 min
p24-s8 kernelsynth bg 40:50 saved; 2.1 min
p24-s8 kernelsynth bg 50:60 saved; 2.3 min
p24-s8 kernelsynth bg 60:70 saved; 2.4 min
p24-s8 kernelsynth bg 70:80 saved; 2.5 min
p24-s8 kernelsynth bg 80:90 sa

,generator,role,n,forecast_hit,true_future_hit,background_true_future_hit,background_forecast_hit,tone_minus_background_forecast,truth_minus_background_truth
0,kernelsynth,hi,185500,0.022453,0.918992,0.016571,0.014733,0.007720,0.902420
1,kernelsynth,lo,185500,0.058280,0.899035,0.025655,0.029240,0.029040,0.873380
2,kernelsynth,lock,185500,0.165105,0.932081,0.033585,0.047391,0.117714,0.898496
3,tsmixup,hi,185500,0.024668,0.886491,0.025563,0.012960,0.011709,0.860927
4,tsmixup,lo,185500,0.066286,0.883229,0.026938,0.027003,0.039283,0.856291
5,tsmixup,lock,185500,0.175935,0.889844,0.025364,0.052226,0.123709,0.864480


,model,role,n,forecast_hit,true_future_hit,background_true_future_hit,background_forecast_hit,tone_minus_background_forecast,truth_minus_background_truth
0,p16-s12,hi,19400,0.037680,0.931649,0.017216,0.028402,0.009278,0.914433
1,p16-s12,lo,19400,0.104897,0.910928,0.027629,0.052268,0.052629,0.883299
2,p16-s12,lock,19400,0.288247,0.921495,0.030773,0.088402,0.199845,0.890722
3,p16-s16,hi,11400,0.003772,0.928246,0.019123,0.007895,-0.004123,0.909123
4,p16-s16,lo,11400,0.015439,0.906754,0.027544,0.022807,-0.007368,0.879211
5,p16-s16,lock,11400,0.151140,0.922719,0.026053,0.072544,0.078596,0.896667
6,p16-s8,hi,11400,0.000088,0.913246,0.014912,0.000877,-0.000789,0.898333
7,p16-s8,lo,11400,0.048860,0.878772,0.030175,0.027807,0.021053,0.848596
8,p16-s8,lock,11400,0.205702,0.922719,0.026053,0.057719,0.147982,0.896667
9,p24-s12,hi,19400,0.049330,0.932113,0.019278,0.029948,0.019381,0.912835


Instrument separation: True | paired background forecasts complete: True


### 4.1 Exact likelihood check
At identical parameters, the count and Bernoulli joint log probabilities differ only by the Binomial combinatorial constant. Unequal phase/trial counts and distinct control roles are retained.


In [13]:
small_arms = arms[arms.model.isin(arms.model.unique()[:3]) & (arms.bg_id<2)].copy()
sort_keys = ["model","P","S","overlap","generator","bg_id","f_lock","role","is_lock"]
small_arms = small_arms.sort_values(sort_keys).reset_index(drop=True)
small_counts = aggregate_trials(small_arms)
bern_model,count_model = model_A2(small_arms,encoding="bernoulli"),model_A2(small_counts)
bern_logp,count_logp = bern_model.compile_logp(),count_model.compile_logp()
point = count_model.initial_point()
constant = np.sum(gammaln(small_counts.n_trials+1)-gammaln(small_counts.hits+1)-gammaln(small_counts.n_trials-small_counts.hits+1))
for shift in (-.6,0.,.4):
    candidate = {key:np.array(value,copy=True) for key,value in point.items()}
    candidate["gamma_config"] = np.linspace(-.8,.6,len(candidate["gamma_config"]))+shift
    candidate["kappa_config"] = np.linspace(-.3,.2,len(candidate["kappa_config"]))
    np.testing.assert_allclose(count_logp(candidate)-bern_logp(candidate),constant,atol=1e-7,rtol=1e-9)
print("A2 Bernoulli/Binomial equivalence: PASS")
del small_arms,small_counts,bern_model,count_model,bern_logp,count_logp
gc.collect()


c:\Users\bonio\dev\patchAliasing\.venv\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


A2 Bernoulli/Binomial equivalence: PASS


79942

## 5. Synthetic parameter recovery
Simulated A2 coefficients include heterogeneous lock and side effects. This is an implementation check, not empirical H1 evidence or a full simulation-based calibration study. The finite-design effect must be covered, at least 80% of listed true coefficients must be covered, and convergence must pass.


In [14]:
recovery_frame,recovery_truth = recovery_data(groups)
recovery_idata = fit_or_load("synthetic_parameter_recovery_A2",recovery_frame,draws=RECOVERY_DRAWS,tune=RECOVERY_TUNE)
ess_threshold = FULL_ESS_MIN if IS_FULL else PILOT_ESS_MIN
recovery_diagnostic,recovery_diagnostics = diagnostics_for(recovery_idata,ess_threshold)
recovery_rows = []
for row in recovery_truth.itertuples(index=False):
    array = recovery_idata.posterior[row.variable]
    if row.coordinate is not None and not pd.isna(row.coordinate): array = array.sel(config=row.coordinate)
    low,median,high = np.quantile(np.asarray(array).ravel(),[.025,.5,.975])
    recovery_rows.append(dict(parameter=row.parameter,truth=row.truth,median=float(median),eti_low=float(low),eti_high=float(high),covered=bool(low<=row.truth<=high)))
recovery_table = pd.DataFrame(recovery_rows)
headline_covered = bool(recovery_table.set_index("parameter").loc["gamma_design","covered"])
RECOVERY_OK = bool(recovery_diagnostic["passed"] and headline_covered and recovery_table.covered.mean()>=RECOVERY_MIN_COVERAGE)
save_table(OUTPUT_ROOT/"synthetic_recovery_summary.parquet",recovery_table)
save_table(OUTPUT_ROOT/"synthetic_recovery_diagnostics.parquet",recovery_diagnostics.reset_index(names="parameter"))
display(recovery_table)
print("SYNTHETIC ONLY:",recovery_diagnostic,"coverage:",recovery_table.covered.mean(),"gate:",RECOVERY_OK)
del recovery_idata,recovery_frame
gc.collect()


synthetic_parameter_recovery_A2: 3,690 groups / 33,390 trials; draws=2000, tune=1500, chains=4, cores=4


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_bar, delta_O, delta_P, tau, beta, gamma_bar, sigma_gamma, gamma_config, kappa_bar, sigma_kappa, kappa_config, sigma_harm, u_harm, sigma_bg, u_bg]


Output()

Sampling 4 chains for 1_500 tune and 2_000 draw iterations (6_000 + 8_000 draws total) took 99 seconds.


Saved synthetic_parameter_recovery_A2.nc in 1.8 minutes


,parameter,truth,median,eti_low,eti_high,covered
0,beta_bar,-0.400000,-0.248192,-0.414921,-0.073842,True
1,delta_O,0.200000,0.033199,-0.308722,0.364149,True
2,delta_P,-0.150000,-0.314011,-0.760376,0.138973,True
3,gamma_bar,-0.250000,-0.301203,-0.532484,-0.071957,True
4,kappa_bar,-0.200000,-0.375924,-0.505365,-0.248500,False
5,gamma_design,-0.345359,-0.318441,-0.376425,-0.261032,True
6,gamma_config[p16-s12],0.427084,0.346609,0.152705,0.546750,True
7,gamma_config[p16-s16],-0.890158,-0.763343,-1.048850,-0.484314,True
8,gamma_config[p16-s8],-0.854387,-0.864700,-1.137333,-0.602463,True
9,gamma_config[p24-s12],-1.196781,-0.872863,-1.073620,-0.655573,False


SYNTHETIC ONLY: {'max_rhat': 1.0019652956501688, 'min_ess_bulk': 5264.4398109321955, 'min_ess_tail': 4883.179913296222, 'divergences': 0, 'passed': True} coverage: 0.9166666666666666 gate: True


133877

## 6. Primary A2 fit
The finite-design effect and per-geometry effects are both reported. An average cannot establish the same behavior in every geometry. Partial or failed convergence remains non-reportable.


In [15]:
idata_A2 = fit_or_load("primary_A2_logit",groups)
primary_diagnostic,parameter_diagnostics = diagnostics_for(idata_A2,ess_threshold)
PRIMARY_OK = primary_diagnostic["passed"]
primary_effect = effect_summary(posterior_array(idata_A2,"gamma_design"))
effect_table = geometry_effect_table(idata_A2,groups)
save_table(OUTPUT_ROOT/"primary_diagnostics.parquet",parameter_diagnostics.reset_index(names="parameter"))
save_table(OUTPUT_ROOT/"geometry_effects.parquet",effect_table)
save_table(OUTPUT_ROOT/"design_effect.parquet",pd.DataFrame([primary_effect]))
display(pd.DataFrame([primary_effect]))
display(effect_table)
print("Primary convergence:",primary_diagnostic)
display(parameter_diagnostics.sort_values("ess_bulk").head(12))


primary_A2_logit: 123,000 groups / 1,113,000 trials; draws=10000, tune=2000, chains=4, cores=4


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_bar, delta_O, delta_P, tau, beta, gamma_bar, sigma_gamma, gamma_config, kappa_bar, sigma_kappa, kappa_config, sigma_harm, u_harm, sigma_bg, u_bg]


Output()

Sampling 2 chains for 2_000 tune and 1_350 draw iterations (4_000 + 2_700 draws total) took 46648 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Saved primary_A2_logit.nc in 777.7 minutes


,median_log_or,eti_low_log_or,eti_high_log_or,median_odds_ratio,p_support,p_equivalence,p_opposite,p_positive
0,3.431105,3.337937,3.562948,30.910795,0.0,0.0,1.0,1.0


,model,median_log_or,eti_low_log_or,eti_high_log_or,median_odds_ratio,p_support,p_equivalence,p_opposite,p_positive,lock_vs_lo_median_log_or,lock_vs_lo_low_log_or,lock_vs_lo_high_log_or,lock_vs_hi_median_log_or,lock_vs_hi_low_log_or,lock_vs_hi_high_log_or,side_median_log_or,side_low_log_or,side_high_log_or
0,p16-s12,2.733423,2.666092,2.800834,15.385458,0.0,0.000000,1.000000,1.0,2.069480,2.000560,2.143518,3.398270,3.307138,3.488035,-0.663761,-0.708761,-0.618172
1,p16-s16,3.666175,3.493858,3.858761,39.102059,0.0,0.000000,1.000000,1.0,2.932779,2.770958,3.103543,4.399445,4.110824,4.722738,-0.733812,-0.907900,-0.571702
2,p16-s8,5.058236,4.553959,5.793360,157.312771,0.0,0.000000,1.000000,1.0,2.221542,2.116823,2.331688,7.894850,6.897783,9.355323,-2.836000,-3.567285,-2.336473
3,p24-s12,3.698292,3.622725,3.774975,40.378265,0.0,0.000000,1.000000,1.0,3.034807,2.957142,3.113182,4.362838,4.265881,4.461329,-0.663513,-0.707581,-0.620411
4,p24-s16,4.997168,4.874635,5.120486,147.993422,0.0,0.000000,1.000000,1.0,4.174836,4.062817,4.284788,5.811824,5.609604,6.025348,-0.817743,-0.934466,-0.706448
5,p24-s20,3.380980,3.300762,3.470206,29.399570,0.0,0.000000,1.000000,1.0,2.864890,2.776053,2.955009,3.899193,3.775270,4.034308,-0.516578,-0.591366,-0.447529
6,p24-s24,6.157694,5.988241,6.336160,472.337397,0.0,0.000000,1.000000,1.0,4.440792,4.343290,4.535692,7.873274,7.569421,8.220067,-1.717544,-1.889165,-1.563445
7,p24-s8,5.972357,5.667511,6.326957,392.429647,0.0,0.000000,1.000000,1.0,3.336642,3.248410,3.427133,8.606255,7.998419,9.306512,-2.636055,-2.983842,-2.330075
8,p32-s12,1.912087,1.854703,1.972472,6.767198,0.0,0.000000,1.000000,1.0,0.910460,0.854395,0.969161,2.914087,2.827307,3.004212,-1.001657,-1.048459,-0.957114
9,p32-s16,2.099170,2.009781,2.190210,8.159395,0.0,0.000000,1.000000,1.0,1.272014,1.183220,1.357277,2.928423,2.788187,3.073386,-0.828836,-0.906727,-0.752251


Primary convergence: {'max_rhat': 1.0178658422410833, 'min_ess_bulk': 36.17301674919067, 'min_ess_tail': 61.355165408594786, 'divergences': 0, 'passed': False}


,r_hat,ess_bulk,ess_tail
u_harm[21.333],1.017578,36.173017,68.323177
u_harm[32.0],1.016951,36.193137,68.564867
u_harm[64.0],1.017296,36.200713,67.719902
beta[p32-s20],1.016845,36.202434,67.860755
u_harm[51.2],1.015440,36.231234,73.254004
u_harm[85.333],1.016542,36.231442,69.491875
beta[p32-s32],1.017323,36.250019,68.589806
beta[p32-s24],1.016710,36.259011,67.591634
beta[p24-s12],1.016464,36.263643,68.549896
u_harm[42.667],1.016726,36.272592,67.280426


### 6.1 Traces and geometry effects
Intervals are 95% equal-tail posterior intervals. The vertical zero line corresponds to equal lock odds and geometric-mean control odds; it is not a claim about pooled control probabilities.


In [ ]:
trace_names = ["gamma_design","gamma_bar","sigma_gamma","kappa_bar","sigma_kappa","beta_bar","tau","sigma_harm","sigma_bg"]
fig,axes = plt.subplots(len(trace_names),1,figsize=(11,15),sharex=True)
for axis,name in zip(axes,trace_names):
    for chain,values in enumerate(np.asarray(idata_A2.posterior[name].transpose("chain","draw"))): axis.plot(values,lw=.55,label=f"chain {chain+1}")
    axis.set_ylabel(name)
axes[0].legend(ncol=CHAINS,fontsize=8)
axes[-1].set_xlabel("retained draw")
fig.suptitle(f"A2, {RUN_MODE}"); fig.tight_layout()
fig.savefig(FIGURE_ROOT/"primary_traces.png",dpi=130); plt.show(); plt.close(fig)
ordered = effect_table.sort_values("median_log_or").reset_index(drop=True)
fig,axis = plt.subplots(figsize=(9,7))
axis.hlines(np.arange(len(ordered)),ordered.eti_low_log_or,ordered.eti_high_log_or,color="tab:blue")
axis.scatter(ordered.median_log_or,np.arange(len(ordered)),s=28,color="tab:blue")
axis.axvline(0,color="black",lw=1); axis.axvline(SUPPORT_LOG_OR,color="tab:red",ls="--",lw=.8)
axis.set_yticks(np.arange(len(ordered)),ordered.model)
axis.set_xlabel("Lock log odds minus midpoint of control log odds")
axis.set_title(f"A2 geometry effects, {RUN_MODE}")
fig.tight_layout(); fig.savefig(FIGURE_ROOT/"geometry_effects.png",dpi=140); plt.show(); plt.close(fig)


### 6.2 PPC: original 34 checks plus separate controls
The original model/generator × lock/control checks remain a separate comparison set. Model/generator × lo/lock/hi adds 51 checks. Each set must reach 90% coverage. These overlapping descriptive checks are not independent tests or model-accuracy percentages. A1 comparison is shown only for identical observations and strata.


In [ ]:
ppc_table = predictive_check(idata_A2,groups)
save_table(OUTPUT_ROOT/"primary_ppc.parquet",ppc_table)
ppc_summary = ppc_table.groupby("check_set").passed.agg(["sum","size","mean"])
PPC_OK = bool(ppc_summary.index.tolist()==["A1_comparable","separate_roles"] and (ppc_summary["mean"]>=PPC_MIN_COVERAGE).all())
display(ppc_summary)
display(ppc_table[~ppc_table.passed])
comparison = comparison_to_A1(ppc_table)
if not comparison.empty:
    save_table(OUTPUT_ROOT/"A1_A2_same_data_ppc.parquet",comparison)
    print("Same-data original checks: A1",comparison.passed_A1.mean(),"A2",comparison.passed_A2.mean())
    display(comparison[["key","passed_A1","passed_A2","observed_rate_A2","rep_low_A2","rep_high_A2"]])
else:
    print("No A1 PPC checkpoint for exactly this data population; no cross-run percentage comparison issued.")
print("A2 PPC gate:",PPC_OK)


## 7. FULL prior and link sensitivity
The primary prior scale is 0.5. FULL adds 0.25, 1.0 and probit with approximately matched prior widths. Probit is converted to the same within-geometry control-log-odds midpoint contrast, then averaged equally over geometries. Up to 2,000 posterior draws are used for this nonlinear conversion. All variants must converge and support/equivalence/opposite probabilities must vary by at most 0.10.


In [ ]:
sensitivity_rows = []
if IS_FULL and PRIMARY_OK:
    sensitivity_rows.append(dict(variant="primary logit .5",passed=PRIMARY_OK,**primary_effect))
    for label,scale,link in (("logit .25",.25,"logit"),("logit 1.0",1.,"logit"),("probit matched .5",.5,"probit")):
        fit = fit_or_load("sensitivity_"+label.replace(" ","_"),groups,scale=scale,link=link)
        diagnostic,_ = diagnostics_for(fit,FULL_ESS_MIN)
        converted = comparable_effects(fit,groups,link=link)
        effect = effect_summary(converted.mean(axis=1))
        sensitivity_rows.append(dict(variant=label,passed=diagnostic["passed"],**effect))
        del fit; gc.collect()
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    columns = ["p_support","p_equivalence","p_opposite"]
    spread = sensitivity_table[columns].max()-sensitivity_table[columns].min()
    SENSITIVITY_OK = bool(sensitivity_table.passed.all() and (spread<=SENSITIVITY_MAX_SPREAD).all())
    save_table(OUTPUT_ROOT/"sensitivity.parquet",sensitivity_table)
    display(sensitivity_table); print("Probability spreads:",spread.to_dict())
else:
    sensitivity_table = pd.DataFrame(); SENSITIVITY_OK = False
    print("Sensitivity deferred: PILOT or failed primary convergence.")


## 8. Final result
PILOT is non-reportable. FULL requires instrument separation, complete paired background forecasts, synthetic recovery, convergence, both PPC sets and sensitivity. An opposite-direction result is allowed. Background-only forecasts inform interpretation; no favorable calibration outcome is required or silently selected.


In [ ]:
PILOT_ROUTE_OK = bool(MEASUREMENT_OK and CALIBRATION_COMPLETE and RECOVERY_OK and PRIMARY_OK and PPC_OK)
FINAL_GATE_OK = bool(IS_FULL and PILOT_ROUTE_OK and SENSITIVITY_OK)
gate_table = pd.DataFrame([dict(gate=name,passed=bool(value)) for name,value in [
    ("instrument separation",MEASUREMENT_OK),("paired background forecasts",CALIBRATION_COMPLETE),
    ("synthetic A2 recovery",RECOVERY_OK),("A2 convergence",PRIMARY_OK),("both A2 PPC sets",PPC_OK),
    ("FULL mode",IS_FULL),("prior/link sensitivity",SENSITIVITY_OK)]])
result = dict(run_mode=RUN_MODE,model_version=MODEL_VERSION,core_fingerprint=CORE_FINGERPRINT,
              analysis_fingerprint=ANALYSIS_FINGERPRINT,pilot_route_ok=PILOT_ROUTE_OK,gate_ok=FINAL_GATE_OK,
              verdict=verdict_from(FINAL_GATE_OK,primary_effect),scope="all_arms",
              estimand="equal-geometry mean log odds versus midpoint of lo/hi log odds",effect=primary_effect,
              primary_diagnostic=primary_diagnostic,gates=gate_table.to_dict("records"))
cp.atomic_json(OUTPUT_ROOT/"final_verdict.json",result); record_artifact(OUTPUT_ROOT/"final_verdict.json")
save_table(OUTPUT_ROOT/"gates.parquet",gate_table)
display(gate_table); print("FINAL A2:",result["verdict"])
if not IS_FULL:
    print("A2 PILOT PASS: change RUN_MODE to FULL in a clean runtime." if PILOT_ROUTE_OK else "A2 PILOT FAIL: inspect the failed gates before FULL.")
print("Artifacts:",OUTPUT_ROOT)


The delivered notebook contains no fitted results. Software checks with synthetic inputs do not establish empirical convergence or validity. Calibration and inference remain conditional on this measurement protocol, signal generators, three pilot backgrounds per generator, and one trained checkpoint per geometry. The notebook preserves the raw forecast-localisation question; it does not silently substitute amplitude recovery or a baseline-subtracted response.
